## 연습 1: 접근 제어

당신은 **NorthMart Retail**의 데이터 엔지니어입니다. 전국 슈퍼마켓 체인입니다. 팀은 고객 및 거래 데이터를 Unity Catalog에 저장합니다. 분석가가 쿼리를 실행하기 전에 먼저 카탈로그, 스키마를 만들고 고객 테이블을 채운 후 `retail-analysts` 그룹에 필요한 권한을 부여해야 합니다.

> ⚠️ 이 연습을 시작하기 전에 Databricks에서 `retail-analysts` 그룹을 만들고 자신의 사용자 계정을 추가했는지 확인하십시오. 실습 설정 지침을 참조하십시오.

### 작업 1.1 — 카탈로그, 스키마 및 고객 테이블 생성

아래 셀은 이 연습에 필요한 모든 객체를 설정합니다:
- 모든 NorthMart 데이터 자산을 보관할 `retail_catalog`라는 카탈로그를 생성합니다.
- 그 안에 `security_lab`이라는 스키마를 생성합니다.
- 관리형 Delta 테이블 `retail_catalog.security_lab.customers`를 생성하고 6개의 샘플 고객 레코드로 채웁니다. 이 행은 행 필터링과 열 마스킹을 시연하기 위해 이 연습의 나머지 부분 전체에서 사용됩니다.

| customer_id | customer_name   | email                   | region | total_spend | tier     |
|-------------|-----------------|-------------------------|--------|-------------|----------|
| 1           | Alice Johnson   | alice@northmart.com     | North  | 5200.00     | Standard |
| 2           | Bob Martinez    | bob@northmart.com       | South  | 8100.00     | Premium  |
| 3           | Carol Lee       | carol@northmart.com     | North  | 3400.00     | Standard |
| 4           | David Chen      | david@northmart.com     | East   | 7800.00     | Premium  |
| 5           | Eve Patel       | eve@northmart.com       | West   | 6200.00     | Standard |
| 6           | Frank Kim       | frank@northmart.com     | South  | 4100.00     | Standard |

셀을 실행하여 이러한 객체를 프로비저닝합니다.

In [ ]:
# Since no default storage is enabled, we are inheriting the storage path from the default catalog's root.
catalogs = [catalog.name for catalog in spark.catalog.listCatalogs("adb_dp750*")]
dp750_default_catalog = catalogs[0] if catalogs else None

storage_root = (
    spark.sql(f"DESCRIBE CATALOG EXTENDED {dp750_default_catalog}")
    .filter("info_name = 'Storage Root'")
    .select("info_value")
    .first()[0]
)
print (f"Storage root: {storage_root}")

spark.sql(f"""
    CREATE CATALOG IF NOT EXISTS retail_catalog
    MANAGED LOCATION '{storage_root}'
    COMMENT 'Catalog for NorthMart Retail data assets'
""")

In [ ]:
%sql
CREATE SCHEMA IF NOT EXISTS retail_catalog.security_lab
  COMMENT 'Schema used for the security lab exercises';

CREATE OR REPLACE TABLE retail_catalog.security_lab.customers AS
SELECT * FROM VALUES
  (1, 'Alice Johnson', 'alice@northmart.com', 'North', 5200.00, 'Standard'),
  (2, 'Bob Martinez',  'bob@northmart.com',   'South', 8100.00, 'Premium'),
  (3, 'Carol Lee',     'carol@northmart.com', 'North', 3400.00, 'Standard'),
  (4, 'David Chen',    'david@northmart.com', 'East',  7800.00, 'Premium'),
  (5, 'Eve Patel',     'eve@northmart.com',   'West',  6200.00, 'Standard'),
  (6, 'Frank Kim',     'frank@northmart.com', 'South', 4100.00, 'Standard')
  AS t(customer_id, customer_name, email, region, total_spend, tier);

### 작업 1.2 — retail-analysts 그룹에 권한 부여

`retail-analysts` 그룹은 `customers` 테이블을 쿼리해야 합니다. Unity Catalog 계층의 모든 필수 수준에서 필요한 권한을 부여합니다:

- `retail_catalog`에 `USE CATALOG` 부여
- `retail_catalog.security_lab`에 `USE SCHEMA` 부여
- `retail_catalog.security_lab.customers`에 `SELECT` 부여

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"Databricks 그룹이 Unity Catalog 테이블에서 SELECT를 수행할 수 있도록 하는 데 필요한 SQL 권한은 무엇입니까?"*

**힌트:** `GRANT <privilege> ON <object type> <object name> TO \`retail-analysts\``를 사용하십시오.

In [ ]:
%sql
GRANT USE CATALOG ON CATALOG retail_catalog TO `retail-analysts`;
GRANT USE SCHEMA ON SCHEMA retail_catalog.security_lab TO `retail-analysts`;
GRANT SELECT ON TABLE retail_catalog.security_lab.customers TO `retail-analysts`;

### 작업 1.3 — 부여된 권한 확인

권한이 올바르게 할당되었는지 확인합니다. `SHOW GRANTS`를 사용하여 `customers` 테이블의 모든 권한을 표시합니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"Unity Catalog 테이블의 권한을 확인하기 위해 SHOW GRANTS를 사용하려면 어떻게 해야 합니까?"*

**예상 결과:** `retail-analysts`가 `SELECT` 권한과 함께 나열되어야 합니다.

In [ ]:
%sql
SHOW GRANTS ON TABLE retail_catalog.security_lab.customers;

## 연습 2: 행 필터링

NorthMart Retail의 지역 분석가는 자신의 지역에 대한 고객 레코드만 봐야 합니다. 쿼리하는 사용자를 기반으로 행을 제한하는 **행 필터 함수**를 구현합니다.

여러 사용자 계정을 만들 수 없으므로 모든 사용자를 위해 `North` 지역 행만 표시하되 자신의 사용자 계정을 제외한 필터를 작성합니다. 자신의 사용자 계정은 모든 행을 봅니다. 이는 지역 분석가가 경험하는 것을 시뮬레이션합니다.

### 작업 2.1 — 행 필터 함수 생성

`retail_catalog.security_lab.region_filter` SQL 함수를 생성합니다. `STRING` 유형의 `region` 매개변수를 받고 다음 경우 `TRUE`를 반환합니다:
- `region`이 `'North'`이거나 **또는**
- 현재 사용자가 자신의 계정입니다(`your@email.com`을 실제 이메일로 바꾸십시오).

즉, 권한이 있는 사용자가 아닌 한 `North` 지역 행만 표시됩니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"current_user()를 사용하여 접근 권한을 제어하는 Databricks Unity Catalog의 행 필터 함수를 만들려면 어떻게 해야 합니까?"*

**힌트:** `CREATE OR REPLACE FUNCTION ... RETURN <current_user()를 사용한 부울 표현식>`을 사용하십시오.

In [ ]:
%sql
-- Replace 'your@email.com' with your actual Azure Databricks user email
CREATE OR REPLACE FUNCTION retail_catalog.security_lab.region_filter(region STRING)
RETURN region = 'North' OR current_user() = 'your@email.com';

### 작업 2.2 — 고객 테이블에 행 필터 적용

`region_filter` 함수를 `customers` 테이블에 연결하여 해당 테이블에 대한 모든 쿼리에 자동으로 적용되도록 합니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"Azure Databricks에서 Unity Catalog 테이블에 행 필터를 설정하려면 어떻게 해야 합니까?"*

**힌트:** `ALTER TABLE ... SET ROW FILTER <function> ON (<column>)`을 사용하십시오.

In [ ]:
%sql
ALTER TABLE retail_catalog.security_lab.customers
SET ROW FILTER retail_catalog.security_lab.region_filter ON (region);

### 작업 2.3 — 행 필터 테스트

`customers` 테이블을 쿼리합니다. 필터 함수에서 권한 있는 사용자로 나열된 자신의 이메일이므로 **모든 6개 행**을 봐야 합니다.

다른 사람으로 로그인했다면(자신의 이메일이 아님) `North` 지역의 2개 행만 볼 수 있습니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"SQL을 사용하여 Databricks에서 현재 쿼리를 실행하는 사용자를 확인하려면 어떻게 해야 합니까?"*

In [ ]:
%sql
-- Verify the current user
SELECT current_user();

-- Query the customers table — as the privileged user you should see all 6 rows
SELECT * FROM retail_catalog.security_lab.customers;

➡️ 팁: 이전에 생성한 행 필터 함수에서 이메일 주소를 업데이트하고 다시 테스트합니다. 고객 테이블을 쿼리할 때 출력을 관찰합니다.

### 작업 2.4 — 행 필터 제거

이제 `customers` 테이블에서 행 필터를 제거합니다. 제거한 후 테이블을 다시 쿼리하여 **모든 6개 행**이 사용자 컨텍스트와 관계없이 반환되는지 확인합니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"Unity Catalog 테이블에서 행 필터를 제거하려면 어떻게 해야 합니까?"*

**힌트:** `ALTER TABLE ... DROP ROW FILTER`를 사용하십시오.

In [ ]:
%sql
ALTER TABLE retail_catalog.security_lab.customers DROP ROW FILTER;

SELECT * FROM retail_catalog.security_lab.customers;

## 연습 3: 열 마스킹

고객 이메일 주소는 PII이므로 권한이 없는 사용자에 대해 마스킹되어야 합니다. 대부분의 사용자에 대해 전체 이메일 주소를 숨기고 `customers` 테이블의 `email` 열에 적용하는 **열 마스크 함수**를 만듭니다.

### 작업 3.1 — 열 마스크 함수 생성

`retail_catalog.security_lab.mask_email` SQL 함수를 생성합니다. `STRING` 유형의 `email` 매개변수를 받고:
- 현재 사용자가 자신의 계정이면 **전체 이메일**을 반환합니다.
- 다른 모든 사용자에게 **마스킹된 값**(예: `'***@***.com'`)을 반환합니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"current_user()를 사용하여 실제 또는 마스킹된 값을 반환할지 결정하는 Databricks Unity Catalog의 열 마스킹 함수를 만들려면 어떻게 해야 합니까?"*

**힌트:** 함수 본체 내에 `CASE WHEN current_user() = '...' THEN email ELSE '***@***.com' END` 패턴을 사용하십시오.

In [ ]:
%sql
-- Replace 'your@email.com' with your actual Azure Databricks user email
CREATE OR REPLACE FUNCTION retail_catalog.security_lab.mask_email(email STRING)
RETURN CASE
  WHEN current_user() = 'your@email.com' THEN email
  ELSE '***@***.com'
END;

### 작업 3.2 — 이메일 열에 열 마스크 적용

`mask_email` 함수를 `customers` 테이블의 `email` 열에 대한 열 마스크로 적용합니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"Azure Databricks에서 SQL을 사용하여 Unity Catalog 테이블 열에 열 마스크를 설정하려면 어떻게 해야 합니까?"*

**힌트:** `ALTER TABLE ... ALTER COLUMN email SET MASK <function>`을 사용하십시오.

In [ ]:
%sql
ALTER TABLE retail_catalog.security_lab.customers
ALTER COLUMN email SET MASK retail_catalog.security_lab.mask_email;

### 작업 3.3 — 열 마스크 테스트

`customers` 테이블을 쿼리합니다. 마스크 함수에 지정된 권한이 있는 사용자이므로 **실제 이메일 주소**를 봐야 합니다.

이 테이블을 쿼리하는 다른 사용자는 `email` 열에서 `***@***.com`을 볼 것입니다.

In [ ]:
%sql
-- Your own account sees real emails; any other user sees ***@***.com
SELECT customer_id, customer_name, email, region FROM retail_catalog.security_lab.customers;

### 작업 3.4 — 열 마스크 제거

`email` 열에서 열 마스크를 제거한 후 테이블을 다시 쿼리하여 모든 사용자에게 이메일이 이제 완전히 표시되는지 확인합니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"Unity Catalog 테이블 열에서 열 마스크를 삭제하려면 어떻게 해야 합니까?"*

**힌트:** `ALTER TABLE ... ALTER COLUMN email DROP MASK`를 사용하십시오.

In [ ]:
%sql
ALTER TABLE retail_catalog.security_lab.customers ALTER COLUMN email DROP MASK;

SELECT customer_id, customer_name, email, region FROM retail_catalog.security_lab.customers;

## 연습 4: Azure Key Vault Secrets

NorthMart Retail의 충성도 플랫폼은 연결하려면 API 키가 필요합니다. 이 키를 노트북 내에 일반 텍스트로 저장하는 것은 보안 위험입니다. 이 연습에서는 실습 설정 중에 생성한 Azure Key Vault 기반 보안 범위에서 API 키를 안전하게 검색합니다.

> ⚠️ 이 연습의 셀을 실행하기 전에 실습 지침에서 Key Vault 설정 단계를 완료했는지 확인합니다(Key Vault 생성, 보안 추가 `loyalty-api-key`, 보안 범위 생성 `retail-kv-scope`).

### 작업 4.1 — 범위의 사용 가능한 보안 목록

`retail-kv-scope` 보안 범위에서 사용 가능한 모든 보안 목록을 표시합니다. `loyalty-api-key`가 접근 가능한지 확인할 수 있습니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"dbutils를 사용하여 Databricks 보안 범위의 보안 목록을 표시하려면 어떻게 해야 합니까?"*

**힌트:** `dbutils.secrets.list("scope-name")`을 사용하십시오. 이는 키 이름만 반환합니다. 보안 값은 절대 반환되지 않습니다.

In [ ]:
secrets = dbutils.secrets.list("retail-kv-scope")
for s in secrets:
    print(s.key)

### 작업 4.2 — Python에서 API 키를 안전하게 검색합니다.

`retail-kv-scope` 범위에서 `loyalty-api-key` 보안을 검색하고 변수에 할당합니다. 그런 다음 인쇄합니다.

Azure Databricks가 노트북 출력에서 값을 **자동으로 수정**하는 방식을 관찰합니다. 실제 키 대신 `[REDACTED]`를 볼 것입니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"dbutils.secrets.get을 사용하여 Databricks 보안 범위에서 보안을 검색하려면 어떻게 해야 합니까? 출력이 수정된 이유는 무엇입니까?"*

In [ ]:
api_key = dbutils.secrets.get(scope="retail-kv-scope", key="loyalty-api-key")
print(api_key)  # Output will show [REDACTED]

### 작업 4.3 — 연결 문자열에서 보안 사용

API 키를 포함하는 연결 문자열을 구성합니다. NorthMart의 충성도 플랫폼에 연결하는 방법을 시뮬레이션합니다. 연결 문자열을 인쇄하고 보안 값이 여전히 수정된 것을 관찰합니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"연결 문자열에 값을 노출하지 않고 f-string 내에서 Databricks 보안을 안전하게 사용하려면 어떻게 해야 합니까?"*

In [ ]:
connection_string = f"https://loyalty.northmart.com/api?key={api_key}"
print(connection_string)  # The api_key portion will appear as [REDACTED]

### 작업 4.4 — SQL을 사용하여 보안 검색

`secret()` SQL 함수를 사용하여 동일한 보안을 검색합니다. SQL 기반 연결을 구성할 때 유용합니다.

SQL 결과에서도 출력이 수정된 것을 참고합니다.

> 🤖 **Genie Code 팁:** 다음을 요청하십시오:
> *"Databricks SQL의 secret() 함수를 사용하여 범위에서 보안을 검색하려면 어떻게 해야 합니까?"*

In [ ]:
%sql
SELECT secret('retail-kv-scope', 'loyalty-api-key') AS loyalty_api_key;

## 정리

아래 셀을 실행하여 이 실습 중에 생성된 모든 객체를 제거합니다.

In [ ]:
%sql
-- Remove all lab objects
ALTER TABLE retail_catalog.security_lab.customers DROP ROW FILTER;
ALTER TABLE retail_catalog.security_lab.customers ALTER COLUMN email DROP MASK;
DROP TABLE IF EXISTS retail_catalog.security_lab.customers;
DROP FUNCTION IF EXISTS retail_catalog.security_lab.region_filter;
DROP FUNCTION IF EXISTS retail_catalog.security_lab.mask_email;
DROP SCHEMA IF EXISTS retail_catalog.security_lab;
DROP CATALOG IF EXISTS retail_catalog CASCADE;